# Session 03 — Draw a helpful camera guide

**Today’s new skill:** use drawings on a webcam frame to tell a person what to do.

We already know how a system can find an object. Here, the object is an orange item. Our job is to draw a target and give clear feedback: **READY**, **ADJUST**, or **LOST**.

## Before you start

Run this notebook **locally** in an environment that can use your webcam. Hold any orange object in front of the camera. Press **q** in the camera window when you are finished.

In [ ]:
# Run this once if OpenCV is not installed on your computer.
%pip install opencv-python

# OpenCV lets Python read the webcam and draw on each frame.
import cv2

## 1. Choose the rule you will test

Start with the suggested value. Later, change **only** `READY_TOLERANCE` and observe how the decision changes.

In [ ]:
# Orange colour range. Leave these two lines alone for the first test.
ORANGE_LOW = (4, 120, 90)
ORANGE_HIGH = (24, 255, 255)

# The blue target box in the middle of the camera frame.
TARGET_WIDTH = 360
TARGET_HEIGHT = 240

# ★ YOUR FIRST EDIT: how close must the orange object be to the target centre?
# Try 24 first. Then try 50. Which value gives the most useful feedback?
READY_TOLERANCE = 24

## 2. Find the centre of the orange object

This helper does one job: it returns the centre point of the largest orange shape it can see. If it cannot see one, it returns `None`.

In [ ]:
def find_orange_centre(frame):
    # Change the frame into HSV colours. HSV makes orange easier to isolate.
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Keep only pixels in the orange range. Everything else becomes black.
    orange_mask = cv2.inRange(hsv, ORANGE_LOW, ORANGE_HIGH)

    # Find each orange shape in the mask.
    contours, _ = cv2.findContours(
        orange_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # Stop here if there is no orange shape to use.
    if not contours:
        return None

    # Use the biggest orange shape. This ignores tiny orange specks.
    biggest_shape = max(contours, key=cv2.contourArea)
    if cv2.contourArea(biggest_shape) < 700:
        return None

    # Calculate the middle of the chosen shape.
    moments = cv2.moments(biggest_shape)
    if moments['m00'] == 0:
        return None
    centre_x = int(moments['m10'] / moments['m00'])
    centre_y = int(moments['m01'] / moments['m00'])
    return (centre_x, centre_y)

## 3. Run the live alignment guide

Read the comments while the webcam window is open. The important question is: does the status tell the user the right next action?

In [ ]:
# Start the default webcam: 0 means the first camera on this computer.
camera = cv2.VideoCapture(0)
if not camera.isOpened():
    raise RuntimeError('No webcam available. Run locally and allow camera access.')

while True:
    # Read one new camera frame.
    ok, frame = camera.read()
    if not ok:
        break

    # Work out the middle target box for this frame.
    frame_height, frame_width = frame.shape[:2]
    left = (frame_width - TARGET_WIDTH) // 2
    top = (frame_height - TARGET_HEIGHT) // 2
    right = left + TARGET_WIDTH
    bottom = top + TARGET_HEIGHT
    target_centre = (left + TARGET_WIDTH // 2, top + TARGET_HEIGHT // 2)

    # Draw the guide first: box plus a cross at its centre.
    blue = (21, 138, 194)
    orange = (246, 138, 69)
    cv2.rectangle(frame, (left, top), (right, bottom), blue, 3)
    cv2.drawMarker(frame, target_centre, orange, cv2.MARKER_CROSS, 20, 2)

    # Ask the helper for the orange object's centre point.
    object_centre = find_orange_centre(frame)

    # Choose the message that the user should see.
    status = 'LOST — show orange object'
    status_colour = orange

    if object_centre is not None:
        object_x, object_y = object_centre

        # Measure the straight-line distance from object centre to target centre.
        distance = ((object_x - target_centre[0]) ** 2 +
                    (object_y - target_centre[1]) ** 2) ** 0.5

        # READY needs both conditions: inside the box and close to the centre.
        inside_box = left <= object_x <= right and top <= object_y <= bottom
        ready = inside_box and distance <= READY_TOLERANCE

        # Draw what the system measured.
        cv2.circle(frame, object_centre, 8, blue if ready else orange, -1)
        cv2.line(frame, target_centre, object_centre, blue if ready else orange, 2)

        # Tell the user their next action.
        status = 'READY — capture frame' if ready else 'ADJUST — centre orange object'
        status_colour = blue if ready else orange

    # Put the decision and the editable rule onto the camera frame.
    cv2.putText(frame, status, (18, 35), cv2.FONT_HERSHEY_SIMPLEX, .65, status_colour, 2)
    cv2.putText(frame, f'tolerance: {READY_TOLERANCE}px', (18, frame_height - 18),
                cv2.FONT_HERSHEY_SIMPLEX, .6, (255, 255, 255), 2)

    # Show the finished interface. Press q in this window to stop.
    cv2.imshow('Alignment guide — press q to exit', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Close the webcam safely after the loop ends.
camera.release()
cv2.destroyAllWindows()

## 4. Record evidence

1. With the object **centred**, what status appears?
2. With it **off-centre**, what status appears?
3. When it is **out of view**, what status appears?
4. After changing `READY_TOLERANCE`, does the guide approve too early, too late, or at a useful point?

**Exit evidence:** write one rule you changed, one visible result, and one case where a user could still be confused.